# Bài 13 · Trực quan hoá nâng cao & phản biện biểu đồ

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau notebook này, bạn:

1. Vẽ hình thống kê nhiều chiều bằng **seaborn** (boxplot, histplot-hue, heatmap).
2. Làm **bản đồ choropleth** từ GeoJSON của Inside Airbnb — và né bẫy diện tích.
3. Nếm plotly express cho biểu đồ tương tác.
4. **Phản biện và sửa** ba biểu đồ lỗi kiểu AI hay sinh ra (trục kép, pie 12 lát, hai đơn vị một trục).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", rc={"grid.alpha": 0.3})
XANH, CAM = "#1E93AB", "#E8890C"

BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29"
df = pd.read_csv(f"{BASE}/visualisations/listings.csv")
df = df[df["price"].notna() & (df["price"] > 0)]
print(df.shape)

## 1. seaborn — so phân phối giữa nhóm

In [ ]:
thu_tu = ["Shared room", "Private room", "Entire home/apt", "Hotel room"]
fig, ax = plt.subplots(figsize=(8.6, 3.8))
sns.boxplot(data=df, x="price", y="room_type", order=thu_tu,
            color=XANH, showfliers=False, width=0.55, ax=ax)
ax.set_xscale("log")
ax.set_xlabel("giá (CLP/đêm, thang log)"); ax.set_ylabel("")
ax.set_title("Mỗi loại phòng một tầng giá — nhưng chồng lấn đáng kể", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

Boxplot trả lời thứ mà bảng trung vị giấu mất: **độ rộng** và **độ chồng lấn** của các phân phối.
`showfliers=False` vì outlier đã bàn ở buổi 10 — ở đây muốn nhìn "thân" phân phối.

In [ ]:
# hue: chiều thứ ba bằng màu — so HÌNH DÁNG phân phối giá 2 quận
hai_quan = df[df["neighbourhood"].isin(["Santiago", "Las Condes"])]
fig, ax = plt.subplots(figsize=(8.6, 3.6))
sns.histplot(data=hai_quan, x="price", hue="neighbourhood", log_scale=True,
             element="step", stat="density", common_norm=False,
             palette=[XANH, CAM], ax=ax)
ax.set_title("Las Condes: cả phân phối dịch phải ~1 bậc so với Santiago",
             loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# heatmap: pivot_table (buổi 5) thành màu
quan_lon = df["neighbourhood"].value_counts().head(8).index
pv = (df[df["neighbourhood"].isin(quan_lon)]
      .pivot_table(values="price", index="neighbourhood", columns="room_type",
                   aggfunc="median")[["Entire home/apt", "Private room"]] / 1000)

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pv.sort_values("Entire home/apt", ascending=False),
            annot=True, fmt=".0f", cmap="Blues",
            cbar_kws={"label": "giá trung vị (nghìn CLP)"}, ax=ax)
ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()

## 2. Bản đồ choropleth

In [ ]:
import geopandas as gpd

geo = gpd.read_file(f"{BASE}/visualisations/neighbourhoods.geojson")
kpi = df.groupby("neighbourhood").agg(gia=("price", "median"), n=("price", "size")).reset_index()
ban_do = geo.merge(kpi, on="neighbourhood", how="left")

fig, ax = plt.subplots(figsize=(8, 5.4))
ban_do.plot(column="gia", cmap="Blues", legend=True, edgecolor="#999", linewidth=0.5, ax=ax,
            legend_kwds={"label": "giá trung vị (CLP/đêm)", "shrink": 0.7},
            missing_kwds={"color": "#eee"})
ax.set_axis_off()
ax.set_title("Giá leo dần về đông bắc Santiago", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

⚠️ **Bẫy diện tích**: Lo Barnechea (quận núi khổng lồ, màu đậm nhất) chiếm nửa bản đồ nhưng chỉ có
~800 listing — trong khi quận Santiago nhỏ xíu chứa 7.000+. Mắt cân theo diện tích, không theo n.
Choropleth luôn cần bảng n đi kèm:

In [ ]:
kpi.nlargest(5, "n")[["neighbourhood", "n", "gia"]]

## 3. plotly express — hover là công cụ soi

In [ ]:
import plotly.express as px

s = df.sample(3000, random_state=1)
fig = px.scatter(s, x="longitude", y="latitude", color="room_type",
                 hover_name="name", hover_data={"price": ":,.0f", "neighbourhood": True},
                 opacity=0.55, title="Rê chuột lên từng điểm để soi (3.000 listing mẫu)")
fig.update_layout(height=420)
fig.show()

Tương tác **để khám phá**; báo cáo PDF của bài tập lớn vẫn dùng hình tĩnh (annotation thay hover).
Dashboard HTML (`fig.write_html("dashboard.html")`) là mục điểm thưởng.

## 4. Phòng phản biện: ba hình AI lỗi

Ba cell dưới **tự tay tạo ra ba biểu đồ lỗi** — đúng kiểu AI hay sinh. Việc của bạn ở mục 5:
bắt lỗi và sửa từng cái.

In [ ]:
# Hình lỗi A: trục kép "tương quan dàn dựng"
rv = pd.read_csv(f"{BASE}/visualisations/reviews.csv", parse_dates=["date"])
thang = rv[rv["date"] < "2026-07-01"].set_index("date").resample("ME").size().loc["2024":]
gia_gia_lap = pd.Series(np.linspace(55, 62, len(thang))
                        + np.random.default_rng(3).normal(0, 1.2, len(thang)), index=thang.index)

fig, ax1 = plt.subplots(figsize=(8.6, 3.6))
ax1.plot(thang.index, thang.values, color=XANH, lw=2)
ax1.set_ylabel("số review", color=XANH)
ax2 = ax1.twinx()
ax2.plot(gia_gia_lap.index, gia_gia_lap.values, color="red", ls="--", lw=2)
ax2.set_ylabel("giá trung vị (nghìn CLP)", color="red")
ax2.set_ylim(54, 63)
ax1.set_title("Demand and Price Analysis", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Hình lỗi B: pie 12 lát cầu vồng
quan12 = df["neighbourhood"].value_counts().head(12)
fig, ax = plt.subplots(figsize=(6.6, 4.4))
ax.pie(quan12.values, labels=quan12.index, autopct="%1.1f%%",
       colors=plt.cm.tab20.colors, startangle=90, textprops={"fontsize": 8})
ax.set_title("Distribution of Listings by Neighbourhood", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Hình lỗi C: hai tiền tệ, một trục
idx = pd.date_range("2024-07-31", periods=24, freq="ME")
scl = (np.linspace(52, 65, 24) + np.random.default_rng(1).normal(0, 1.5, 24)) * 1000  # CLP
rio = np.linspace(280, 340, 24) + np.random.default_rng(2).normal(0, 8, 24)           # BRL

fig, ax = plt.subplots(figsize=(8.6, 3.4))
ax.plot(idx, scl, lw=2, color=XANH, label="Santiago (CLP)")
ax.plot(idx, rio, lw=2, color=CAM, label="Rio (BRL)")
ax.set_title("Price Comparison: Santiago vs Rio", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

## 5. Bài tập tại lớp — hồ sơ lỗi & bản sửa

### Bài 1 — Lập hồ sơ lỗi

Với **mỗi** hình lỗi, viết vào cell Markdown dưới: (a) hình mời bạn tin điều gì,
(b) thủ thuật thị giác nào tạo ra ấn tượng đó, (c) một câu chất vấn kiểu vấn đáp.
Dùng checklist 5 câu: *trục — đơn vị — n — dạng hình — lời diễn giải*.

*(Viết hồ sơ lỗi của bạn vào đây — double-click để sửa)*

- **Hình lỗi A:** …
- **Hình lỗi B:** …
- **Hình lỗi C:** …

### Bài 2 — Sửa hình lỗi B

Vẽ lại pie 12 lát thành **bar ngang xếp hạng** chuẩn buổi 12 (nhấn 1 thanh, nhãn %,
title thông điệp).

In [ ]:
# TODO Bài 2:
ty_le = (quan12 / len(df) * 100).sort_values()
fig, ax = plt.subplots(figsize=(8, 4.2))
mau = [CAM if q == ty_le.idxmax() else XANH for q in ty_le.index]
bars = ax.barh(ty_le.index, ty_le.values, color=mau, height=0.6)
ax.bar_label(bars, [f" {v:.1f}%" for v in ty_le.values], fontsize=9)
ax.set_title("2 phần 5 số listing nằm ở quận trung tâm Santiago", loc="left", fontweight="bold")
ax.set_xlabel("% tổng listing (n = {:,})".format(len(df)))
ax.grid(axis="y", alpha=0)
plt.tight_layout(); plt.show()

### Bài 3 — Sửa hình lỗi C

Quy hai chuỗi giá về **chỉ số tháng đầu = 100** rồi vẽ lại trên MỘT trục (giờ thì hợp lệ,
vì cùng không thứ nguyên). Kết luận nào rút ra được từ bản sửa mà bản gốc không cho phép?

In [ ]:
# TODO Bài 3:
fig, ax = plt.subplots(figsize=(8.6, 3.4))
ax.plot(idx, scl / scl[0] * 100, lw=2.2, color=XANH, label="Santiago")
ax.plot(idx, rio / rio[0] * 100, lw=2.2, color=CAM, label="Rio")
ax.axhline(100, color="#777", lw=1, ls="--")
ax.set_ylabel("chỉ số giá (tháng đầu = 100)")
ax.set_title("Quy về mốc chung: Santiago tăng nhanh hơn Rio", loc="left", fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 6. Thử thách về nhà 🏆 — Choropleth cho thành phố của nhóm

1. Tải `neighbourhoods.geojson` + `visualisations/listings.csv` của thành phố nhóm bạn nhận cho bài tập lớn.
2. Vẽ 2 bản đồ cạnh nhau: (a) giá trung vị theo quận, (b) **số listing** theo quận —
   cặp bản đồ này "giải độc" lẫn nhau (giá trị vs khối lượng).
3. Một đoạn 5 câu: khu nào đắt, khu nào đông, hai bản đồ kể chuyện gì khác nhau,
   và bẫy diện tích ảnh hưởng thế nào ở thành phố của bạn.

In [ ]:
RUN_CHALLENGE = False
if RUN_CHALLENGE:
    CITY_BASE = "https://data.insideairbnb.com/..."
    ...

---

## Tóm tắt buổi học

| Ý chốt | Vì sao quan trọng |
|---|---|
| seaborn: data= + tên cột; box/hue/heatmap | So phân phối giữa nhóm trong một hình |
| Choropleth = geo + KPI + merge; kèm n | Bản đồ đẹp nhất cũng thua bẫy diện tích |
| Tương tác khám phá, tĩnh xuất bản | Đúng công cụ đúng khán giả |
| Checklist 5 câu: trục, đơn vị, n, dạng, lời | Vũ khí phản biện mọi hình — kể cả hình AI |

**Buổi sau:** ghép tất cả thành **câu chuyện dữ liệu** — và thẩm định trọn một bản phân tích do AI viết.